# ECG画像分類：Google Colab 推論例

Takotsubo（`tako`）と acute anterior STEMI（`AMI`）を分類する、研究用の推論例です。ローカルCLIと同じコードを呼び出します。

既定値は **ViT: TTA 24 / magnitude 1、ConvNeXtV2: TTAなし、MLP-Mixer: TTA 32 / magnitude 3**。アンサンブルは各モデルの **raw logitsを等重みで平均した後、softmax** を適用します。TTA回数には変換なしの1回を含みます。設定は下のセルで変更できます。

学習済み重み・実患者画像はこのnotebookに含まれていません。推論用に整理した重みと、Colabでの利用が認められた画像を用意してください。公開用の患者画像は不要です。既存の論文予測との一致確認には、対応する非公開画像・予測結果を別途使用します。

## 1. 使用するコードの版を指定する

初回は、このセキュリティ修正が公開された後の `GIT_REF = "main"` を使用できます。再現用には、この修正を含むタグまたはcommit SHAに変更してください。実際にチェックアウトしたSHAを結果と一緒に記録します。

このnotebookには **Python 3.11以上** と、セキュリティ対策を含む新しい推論コードが必要です。`v0.1.1`にはCLIがなく、以前の推論版にも今回の対策は含まれていません。


In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import json
import os
import re
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11以上のColabランタイムを選択してください。")

ENV_DIR = Path("/content/ecg_inference_environment")

REPO_URL = "https://github.com/ekagawa007/takotsubo_ami_ai_ecg.git"
GIT_REF = "main"  # 確認後はcommit SHAまたは対応するReleaseタグに固定
REPO_DIR = Path("/content/takotsubo_ami_ai_ecg")

if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_./-]*", GIT_REF):
    raise ValueError("GIT_REFにブランチ名、タグ名、またはcommit SHAを指定してください。")

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args], text=True
    ).strip()

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError("REPO_DIRが既存のGit作業フォルダではありません。別の保存先を指定してください。")
    if git("remote", "get-url", "origin").rstrip("/") != REPO_URL.rstrip("/"):
        raise RuntimeError("REPO_DIRのリポジトリがREPO_URLと一致しません。")
    if git("status", "--porcelain"):
        raise RuntimeError("作業フォルダに変更があります。保存してから再実行してください。")

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
CODE_COMMIT = git("rev-parse", "HEAD")
required_code = [
    REPO_DIR / "examples" / "predict_ensemble.py",
    REPO_DIR / "requirements-inference.txt",
    REPO_DIR / "src" / "takotsubo_ami_ai_ecg" / "security.py",
]
if any(not path.is_file() for path in required_code):
    raise RuntimeError(
        "指定した版には必要なセキュリティ対策がありません。修正を含むGIT_REFを指定してください。"
    )
print("Code commit:", CODE_COMMIT)
sys.path.insert(0, str(REPO_DIR / "src"))


## 2. 推論用ライブラリを準備する

GPUを使う場合は、Colabのランタイム設定でGPUを選択してください。CPUでも実行できますが、TTAを含む3モデルの推論には時間がかかります。

Colabに入っているPyTorch・torchvisionの組を保持して、推論用の依存関係を追加します。Pillowは修正済みの13.0.0を使用します。学習環境の履歴であるルートの `requirements.txt` は、このセルでは使用しません。

**インストールでライブラリの版が変わった場合は、ここで一度「ランタイム → セッションを再起動」を選び、最初のコードセルから順に実行してください。** 再起動を案内する例外は、この段階で停止するためのものです。再起動せず同じセルを繰り返しても解除されません。Pythonの変数は再起動で消えるため、途中のセルだけを再実行しないでください。


In [ ]:
ENVIRONMENT_READY = False
ENV_DIR.mkdir(parents=True, exist_ok=True)
restart_marker = ENV_DIR / "restart_required.json"


def check_session_restart():
    if restart_marker.is_file():
        recorded = json.loads(restart_marker.read_text(encoding="utf-8"))
        if recorded["pid"] == os.getpid():
            raise RuntimeError(
                "ライブラリを変更しました。「ランタイム → セッションを再起動」を選び、"
                "最初のコードセルから順に実行してください。"
            )
        restart_marker.unlink()


def installed_versions():
    return {
        dist.metadata["Name"].lower().replace("_", "-"): dist.version
        for dist in metadata.distributions()
        if dist.metadata.get("Name")
    }


try:
    torch_version = metadata.version("torch")
    torchvision_version = metadata.version("torchvision")
except metadata.PackageNotFoundError as exc:
    raise RuntimeError("PyTorchとtorchvisionがあるColabランタイムを選択してください。") from exc

constraints_path = ENV_DIR / "colab_torch_constraints.txt"
constraints_path.write_text(
    f"torch=={torch_version}\ntorchvision=={torchvision_version}\n", encoding="utf-8"
)
before_install = installed_versions()
installation = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-inference.txt"),
     "-c", str(constraints_path)],
    check=False,
)
after_install = installed_versions()
changed_packages = sorted(
    name for name in before_install.keys() | after_install.keys()
    if before_install.get(name) != after_install.get(name)
)
if changed_packages:
    restart_marker.write_text(
        json.dumps({"pid": os.getpid(), "changed_packages": changed_packages}), encoding="utf-8"
    )
    print("Changed packages:", ", ".join(changed_packages))
if installation.returncode != 0:
    raise RuntimeError(
        "依存ライブラリのインストールに失敗しました。上のpipのエラーを確認してください。"
        + (" ライブラリの版が変わったため、再試行前にセッションを再起動してください。" if changed_packages else "")
    )
check_session_restart()
ENVIRONMENT_READY = True
print("Preserved PyTorch / torchvision:", torch_version, "/", torchvision_version)


In [ ]:
if not globals().get("ENVIRONMENT_READY", False):
    raise RuntimeError("最初のコードセルから依存ライブラリ準備まで順に実行してください。")
check_session_restart()

from takotsubo_ami_ai_ecg.security import require_patched_pillow
require_patched_pillow()

import torch
import torchvision
from PIL import Image

runtime_versions = {"python": sys.version.split()[0]}
for package in ("torch", "torchvision", "timm", "transformers", "numpy", "Pillow", "safetensors"):
    runtime_versions[package] = metadata.version(package)
runtime_versions["cuda_available"] = torch.cuda.is_available()
runtime_versions["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print(json.dumps(runtime_versions, ensure_ascii=False, indent=2))


## 3. 重みと画像を指定する

推論用のファイル構成は次のとおりです。通常の学習checkpointを直接アップロードせず、`tools/export_inference_weights.py`で作成したファイルを使用してください。

- `weights/vit/model.safetensors` と `weights/vit/config.json`
- `weights/convnextv2/model.safetensors`
- `weights/mlpmixer/model.safetensors`

ViTの `preprocessor_config.json` は設定の記録として一緒に保管できます。元のデータセット評価コードに合わせ、推論では3モデルともImageNetのmean/stdで正規化します。ConvNeXtV2は `ckpt_0011.pth` の通常のモデル重みを使用し、SWAの重みは使用しません。

画像と重みはColabのファイルパネルから配置するか、必要な場合だけ次のセルでGoogle Driveを接続します。Drive内のファイルは入力として読み込み、結果は既定で `/content/ecg_inference_results` に保存します。

In [ ]:
USE_GOOGLE_DRIVE = False  # Driveから読み込む場合だけTrueに変更
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

In [ ]:
# "ensemble", "vit", "convnextv2", "mlpmixer" のいずれか
MODEL = "ensemble"

# Colabに配置したファイルの一般的なパス。手元の配置に合わせて変更してください。
WEIGHTS_DIR = Path("/content/weights")
IMAGE_PATH = Path("/content/data/example.png")
# Driveを使用する場合の例:
# WEIGHTS_DIR = Path("/content/drive/MyDrive/ecg_inference/weights")
# IMAGE_PATH = Path("/content/drive/MyDrive/ecg_inference/data/example.png")

OUTPUT_DIR = Path("/content/ecg_inference_results")
# 出力ファイル名は推論セルの実行ごとに自動生成します。

# 論文のTTA条件。TTA=1は変換なしのみ。
VIT_TTA = 24
VIT_MAGNITUDE = 1
CONVNEXTV2_TTA = 1
CONVNEXTV2_MAGNITUDE = 5  # TTA=1では使用されない
MLPMIXER_TTA = 32
MLPMIXER_MAGNITUDE = 3
ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0]  # ViT, ConvNeXtV2, MLP-Mixerの順

THRESHOLD = None  # None: ensembleはAMI>=0.5、単一モデルはargmax。数値で変更可能
SEED = 1234
DEVICE = "auto"  # "auto", "cpu", "cuda"
PRECISION = "float32"  # "float32" または "bfloat16"
INPUT_POLICY = "strict480"  # "resize" を明示すると480x480にリサイズ
BATCH_SIZE = 1
NUM_WORKERS = 4  # 元の評価設定。必要な場合は0に変更可能

## 4. 入力を確認して推論する

`strict480` は、元のデータセット評価と同じ **480×480画素** の入力を要求します。異なるサイズの場合は学習時と同じ前処理を確認してください。`resize` は明示的な代替設定であり、論文時の画像作成処理との一致を保証するものではありません。

TTAの乱数はseedで制御します。ただし、画像順・バッチサイズ・worker数・ライブラリや精度が異なる場合、当時の一括評価と完全には一致しないことがあります。
推論セルを再実行すると、時刻と識別用文字列を含む新しい出力名を作成します。TTAなしと論文TTAの結果を、それぞれ別のJSONと実行記録として保存できます。


In [ ]:
check_session_restart()
require_patched_pillow()

supported_models = ("ensemble", "vit", "convnextv2", "mlpmixer")
if MODEL not in supported_models:
    raise ValueError(f"MODELは{supported_models}から選択してください。")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError("IMAGE_PATHに入力画像を配置し、パスを指定してください。")

selected_models = ("vit", "convnextv2", "mlpmixer") if MODEL == "ensemble" else (MODEL,)
required_weights = [WEIGHTS_DIR / model / "model.safetensors" for model in selected_models]
if "vit" in selected_models:
    required_weights.append(WEIGHTS_DIR / "vit" / "config.json")
if any(not path.is_file() for path in required_weights):
    raise FileNotFoundError("選択したモデルの推論用重み・設定が不足しています。上記の構成を確認してください。")

with Image.open(IMAGE_PATH) as image:
    input_size = image.size
if INPUT_POLICY == "strict480" and input_size != (480, 480):
    raise ValueError(f"入力は{input_size}画素です。strict480には480x480の画像が必要です。")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input dimensions:", input_size)
print("Selected model:", MODEL)

In [ ]:
result = None

from datetime import datetime, timezone
import hashlib
import uuid

check_session_restart()
require_patched_pillow()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_JSON = OUTPUT_DIR / f"prediction_{MODEL}_{stamp}_{uuid.uuid4().hex[:8]}.json"
RUN_JSON = OUTPUT_JSON.with_suffix(".colab_run.json")

command = [
    sys.executable,
    str(REPO_DIR / "examples" / f"predict_{MODEL}.py"),
    "--image", str(IMAGE_PATH),
    "--weights-dir", str(WEIGHTS_DIR),
    "--output", str(OUTPUT_JSON),
    "--vit-tta", str(VIT_TTA), "--vit-magnitude", str(VIT_MAGNITUDE),
    "--convnextv2-tta", str(CONVNEXTV2_TTA), "--convnextv2-magnitude", str(CONVNEXTV2_MAGNITUDE),
    "--mlpmixer-tta", str(MLPMIXER_TTA), "--mlpmixer-magnitude", str(MLPMIXER_MAGNITUDE),
    "--ensemble-weights", *[str(value) for value in ENSEMBLE_WEIGHTS],
    "--seed", str(SEED),
    "--device", DEVICE,
    "--precision", PRECISION,
    "--input-policy", INPUT_POLICY,
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
if THRESHOLD is not None:
    command.extend(["--threshold", str(THRESHOLD)])
print("Running inference; paper TTA settings may take several minutes.")
completed = subprocess.run(command, cwd=REPO_DIR, capture_output=True, text=True, check=False)
if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(
        f"推論が終了コード {completed.returncode} で停止しました。上に表示されたエラーを確認してください。"
    )
result = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
# 出力と実行記録を対応付ける。入力の絶対パスは記録しない。
run_record = {
    "code_commit": CODE_COMMIT,
    "prediction_file": OUTPUT_JSON.name,
    "prediction_sha256": hashlib.sha256(OUTPUT_JSON.read_bytes()).hexdigest(),
    "runtime": runtime_versions,
    "inference_versions": result["metadata"]["versions"],
    "actual_device": result["metadata"]["actual_device"],
}
with RUN_JSON.open("x", encoding="utf-8") as stream:
    json.dump(run_record, stream, ensure_ascii=False, indent=2)
    stream.write("\n")
print("Inference completed.")
print("Prediction:", OUTPUT_JSON.name)
print("Run record:", RUN_JSON.name)


## 5. 結果を確認する

クラス順は **0 = tako、1 = AMI** です。既定の判定は、アンサンブルでは `prob_AMI >= 0.5`、単一モデルではargmaxです（確率が同率の場合はそれぞれAMI／tako）。`THRESHOLD`に数値を指定すると、どちらもその閾値で判定します。以下にはモデル出力を表示します。正解ラベルとの比較を行っていないため、精度・感度・特異度を示すものではありません。

JSONには実際の実行条件、重みの識別情報、logitsと確率が保存されます。Colabの作業領域は永続保存ではないため、必要な結果を保存してください。
推論が成功すると、`prediction_…json` と、それに対応する `prediction_….colab_run.json` が作成されます。後者にはコードのcommit、実行環境、予測ファイルのSHA-256を記録します。下のダウンロードセルで両方を保存してください。


In [ ]:
if not isinstance(globals().get("result"), dict):
    raise RuntimeError("推論が正常終了してから結果を確認してください。")

for prediction in result["predictions"]:
    summary = {
        "sample_id": prediction.get("sample_id"),
        "pred_id": prediction["pred_id"],
        "pred_label": prediction["pred_label"],
        "logits": prediction["logits"],
        "probabilities": prediction["probabilities"],
        "models": prediction.get("models", {}),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# 必要な場合だけTrueに変更して、同じ実行の2つのJSONをダウンロード
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    if not isinstance(globals().get("result"), dict) or not OUTPUT_JSON.is_file() or not RUN_JSON.is_file():
        raise RuntimeError("推論が正常終了し、予測と実行記録が保存されていることを確認してください。")
    from google.colab import files
    files.download(str(OUTPUT_JSON))
    files.download(str(RUN_JSON))


## 検証範囲

2026-10-22に、修正前の推論コードでは、WSLの実ECG10画像についてfloat32・TTAなしの旧評価コードとの一致を確認し、ColabのA100で合成画像1枚を論文TTA設定・float32で実行しました。今回のPillow更新と読み込み制限を含む版は、これらと区別して検証状況を記録します。最新の状況は `docs/implementation_status.md` を参照してください。

論文の既存予測との照合には、同じ重み、同じ入力画像、同じ前処理を使用します。過去のTTAあり予測との数値的一致は、これまでの確認だけでは確定していません。

このnotebookの配布版には実患者画像・実行結果を保存していません。実行済みnotebookを共有する場合も、画像・予測・ログなどの出力を消去してから共有してください。
